<a href="https://colab.research.google.com/github/Santiago-Echeverri-Arteaga/Fisica_Computacional_2/blob/master/curso_2026_2/01_nivelacion_ml/15_explicabilidad_local_global.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg"
       alt="Abrir en Colab"/>
</a>

# Explicabilidad local y global

**Pregunta guía:** ¿Qué aprendió un modelo y cuándo no debemos creer la explicación?<br>
**Duración sugerida:** 4 horas.<br>
**Entorno:** CPU; datos incluidos o generados en memoria.

El orden de trabajo es siempre: problema → matemática → implementación
mínima → biblioteca → evaluación → interpretación física.


## Tres aproximaciones

1. **Importancia por permutación:** se rompe una variable y se mide la
   caída de desempeño. Es global y refleja dependencia predictiva, no
   causalidad.
2. **Sustituto global:** un modelo interpretable $g(x)$ aproxima las
   predicciones de una caja negra $f(x)$ en toda la población.
3. **Sustituto local:** se perturba alrededor de $x_0$ y se ajusta un
   modelo simple ponderado por proximidad. La fidelidad es local.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import make_friedman1
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import PartialDependenceDisplay, permutation_importance
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor, plot_tree

SEMILLA = 42
X_array, y = make_friedman1(n_samples=1_500, n_features=10, noise=1.0, random_state=SEMILLA)
nombres = [f"sensor_{i}" for i in range(X_array.shape[1])]
X = pd.DataFrame(X_array, columns=nombres)
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEMILLA
)
caja_negra = RandomForestRegressor(
    n_estimators=300, min_samples_leaf=3, random_state=SEMILLA, n_jobs=-1
).fit(X_dev, y_dev)
pred = caja_negra.predict(X_test)
print(f"RMSE test: {root_mean_squared_error(y_test, pred):.3f}")


In [ ]:
perm = permutation_importance(
    caja_negra,
    X_test,
    y_test,
    scoring="neg_root_mean_squared_error",
    n_repeats=15,
    random_state=SEMILLA,
    n_jobs=-1,
)
importancia = pd.DataFrame(
    {"variable": nombres, "caída_media": perm.importances_mean, "std": perm.importances_std}
).sort_values("caída_media", ascending=False)
display(importancia)

variables = importancia.head(3)["variable"].tolist()
PartialDependenceDisplay.from_estimator(caja_negra, X_test, variables)
plt.suptitle("Dependencia parcial: promedio global condicionado por el modelo")
plt.tight_layout()
plt.show()


In [ ]:
# Sustituto global: aprende a reproducir f(X), no las etiquetas originales.
y_caja_dev = caja_negra.predict(X_dev)
sustituto_global = DecisionTreeRegressor(max_depth=3, random_state=SEMILLA).fit(
    X_dev, y_caja_dev
)
fidelidad_global = r2_score(pred, sustituto_global.predict(X_test))
print(f"Fidelidad global R²(g(X), f(X)): {fidelidad_global:.3f}")

plt.figure(figsize=(16, 6))
plot_tree(sustituto_global, feature_names=nombres, filled=True, fontsize=8)
plt.title("Árbol sustituto global")
plt.show()


In [ ]:
def sustituto_local(modelo, x0, X_referencia, n=1500, ancho=0.75, semilla=42):
    rng = np.random.default_rng(semilla)
    escala = X_referencia.std(axis=0).to_numpy()
    vecinos = x0.to_numpy() + rng.normal(size=(n, len(x0))) * escala * ancho
    vecinos = pd.DataFrame(vecinos, columns=X_referencia.columns)
    dist2 = np.sum(((vecinos - x0) / escala) ** 2, axis=1)
    pesos = np.exp(-dist2 / (2 * ancho**2))
    objetivo_local = modelo.predict(vecinos)
    local = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
    local.fit(vecinos, objetivo_local, ridge__sample_weight=pesos)
    coef = local.named_steps["ridge"].coef_
    fidelidad = r2_score(objetivo_local, local.predict(vecinos), sample_weight=pesos)
    return pd.Series(coef, index=X_referencia.columns), fidelidad


x0 = X_test.iloc[0]
coeficientes, fidelidad_local = sustituto_local(caja_negra, x0, X_dev)
print(f"Predicción explicada: {caja_negra.predict(x0.to_frame().T)[0]:.3f}")
print(f"Fidelidad local ponderada: {fidelidad_local:.3f}")
display(coeficientes.reindex(coeficientes.abs().sort_values(ascending=False).index).head(6))


Una explicación no convierte correlación en mecanismo físico. Debe
reportarse junto con su fidelidad, región de validez, variables
correlacionadas y sensibilidad a la semilla/ancho local.

**Ejercicios:** explique tres observaciones distintas; cambie el ancho;
duplique un sensor correlacionado y observe la importancia; compare el
sustituto con la ecuación conocida de `make_friedman1`.
